# ML Model Quality: Eliminating False Positives in Anomaly Detection

**Prerequisite:** Notebook 01 (Anomaly Detection) must be run first  
**Focus:** Training pipeline quality, similarity scores, outlier identification  
**Snowflake Features:** `AI_SIMILARITY`, `AI_EMBED`, `SNOWFLAKE.ML.ANOMALY_DETECTION`, `CORTEX.COMPLETE`

In [ ]:
USE DATABASE CDSB_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
!pip install plotly --quiet

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print('Connected to', session.get_current_database(), session.get_current_schema())

## 1. The False Positive Problem

Anomaly detection models flag unusual patterns — but not all anomalies are **real problems**.  
Some are caused by:
- **Noisy training data** — data entry errors, one-off events
- **Seasonal patterns** the model hasn't learned (e.g. holiday periods)
- **Structural changes** — new suburbs, boundary changes, policy shifts

This notebook demonstrates techniques to **identify and eliminate false positives** in the training pipeline.

In [ ]:
df_anomalies = session.sql("""
    SELECT r.*, s.REGISTRATIONS as ACTUAL
    FROM TMR_ANOMALY_RESULTS r
    JOIN TMR_SUBURB_MONTHLY s ON r.TS = s.TS AND r.SERIES = s.SERIES_ID
    ORDER BY IS_ANOMALY DESC, PERCENTILE DESC
""").to_pandas()

n_total = len(df_anomalies)
n_anomalies = df_anomalies['IS_ANOMALY'].sum()
print(f'Total predictions: {n_total}')
print(f'Flagged anomalies: {n_anomalies} ({100*n_anomalies/n_total:.1f}%)')
print(f'\nTop 10 anomalies by confidence:')
df_anomalies[df_anomalies['IS_ANOMALY'] == True][['SERIES','TS','ACTUAL','FORECAST','PERCENTILE']].head(10)

## 2. Similarity Scores for Outlier Detection

`AI_SIMILARITY` computes cosine similarity between text embeddings.  
We use it to find **anomalous suburbs that are textually similar** to known-good patterns,  
indicating potential false positives.

In [ ]:
df_sim = session.sql("""
    WITH anomalous_suburbs AS (
        SELECT DISTINCT SERIES as SUBURB
        FROM TMR_ANOMALY_RESULTS
        WHERE IS_ANOMALY = TRUE
    ),
    suburb_profiles AS (
        SELECT
            SERIES_ID as SUBURB,
            'Suburb ' || SERIES_ID || ' with average monthly registrations of ' ||
            ROUND(AVG(REGISTRATIONS), 1) || ', std dev ' ||
            ROUND(STDDEV(REGISTRATIONS), 1) || ', over ' ||
            COUNT(*) || ' months, min ' || MIN(REGISTRATIONS) ||
            ' max ' || MAX(REGISTRATIONS) as PROFILE_TEXT
        FROM TMR_SUBURB_MONTHLY
        GROUP BY SERIES_ID
    ),
    baseline_profile AS (
        SELECT
            'Typical QLD suburb with average monthly registrations of ' ||
            ROUND(AVG(REGISTRATIONS), 1) || ', std dev ' ||
            ROUND(STDDEV(REGISTRATIONS), 1) || ', stable seasonal pattern' as BASELINE
        FROM TMR_SUBURB_MONTHLY
    )
    SELECT
        p.SUBURB,
        ROUND(AI_SIMILARITY(p.PROFILE_TEXT, b.BASELINE), 4) as SIMILARITY_TO_BASELINE,
        p.PROFILE_TEXT
    FROM suburb_profiles p
    CROSS JOIN baseline_profile b
    WHERE p.SUBURB IN (SELECT SUBURB FROM anomalous_suburbs)
    ORDER BY SIMILARITY_TO_BASELINE DESC
""").to_pandas()

print('Anomalous suburbs ranked by similarity to typical baseline:')
print(f'HIGH similarity = likely false positive (suburb looks normal)')
print(f'LOW similarity = likely true anomaly (suburb is genuinely unusual)\n')
print(df_sim[['SUBURB','SIMILARITY_TO_BASELINE']].to_string(index=False))

In [ ]:
fig = px.bar(df_sim.sort_values('SIMILARITY_TO_BASELINE', ascending=True),
             x='SIMILARITY_TO_BASELINE', y='SUBURB', orientation='h',
             title='Anomalous Suburbs: Similarity to Typical Baseline',
             color='SIMILARITY_TO_BASELINE',
             color_continuous_scale='RdYlGn')
fig.add_vline(x=df_sim['SIMILARITY_TO_BASELINE'].median(), line_dash='dash',
              annotation_text='Median', line_color='grey')
fig.update_layout(template='plotly_white', yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. Training Data Quality Assessment

Before retraining, we assess the **quality of training data** using statistical methods:  
- **IQR method** — flag series with extreme outliers in training data  
- **Coefficient of variation** — identify highly volatile series that may degrade model accuracy  
- **Short series** — flag suburbs with insufficient history

In [ ]:
df_quality = session.sql("""
    WITH series_stats AS (
        SELECT
            SERIES_ID,
            COUNT(*) as N_MONTHS,
            AVG(REGISTRATIONS) as MEAN_REG,
            STDDEV(REGISTRATIONS) as STD_REG,
            MEDIAN(REGISTRATIONS) as MEDIAN_REG,
            PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY REGISTRATIONS) as Q1,
            PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY REGISTRATIONS) as Q3,
            MIN(REGISTRATIONS) as MIN_REG,
            MAX(REGISTRATIONS) as MAX_REG
        FROM TMR_SUBURB_MONTHLY
        WHERE TS < '2026-02-01'
        GROUP BY SERIES_ID
        HAVING COUNT(*) >= 8
    )
    SELECT
        SERIES_ID,
        N_MONTHS,
        ROUND(MEAN_REG, 1) as MEAN_REG,
        ROUND(STD_REG, 1) as STD_REG,
        ROUND(CASE WHEN MEAN_REG > 0 THEN STD_REG / MEAN_REG ELSE 0 END, 3) as CV,
        ROUND(Q1, 1) as Q1, ROUND(Q3, 1) as Q3,
        ROUND(Q3 - Q1, 1) as IQR,
        CASE
            WHEN MAX_REG > Q3 + 1.5 * (Q3 - Q1) OR MIN_REG < Q1 - 1.5 * (Q3 - Q1)
            THEN TRUE ELSE FALSE
        END as HAS_OUTLIERS,
        CASE
            WHEN MEAN_REG > 0 AND STD_REG / MEAN_REG > 0.5 THEN 'HIGH_VOLATILITY'
            WHEN N_MONTHS < 10 THEN 'SHORT_SERIES'
            ELSE 'CLEAN'
        END as QUALITY_FLAG
    FROM series_stats
    ORDER BY CV DESC
""").to_pandas()

quality_counts = df_quality['QUALITY_FLAG'].value_counts()
outlier_counts = df_quality['HAS_OUTLIERS'].value_counts()
print('Training Data Quality Assessment:')
print(f'Total series: {len(df_quality)}')
print(f'\nQuality Flags:')
for flag, count in quality_counts.items():
    print(f'  {flag}: {count} ({100*count/len(df_quality):.1f}%)')
print(f'\nSeries with IQR outliers: {outlier_counts.get(True, 0)}')
print(f'\nTop 15 most volatile training series (highest CV):')
df_quality[['SERIES_ID','N_MONTHS','MEAN_REG','STD_REG','CV','HAS_OUTLIERS','QUALITY_FLAG']].head(15)

In [ ]:
fig = px.scatter(df_quality, x='MEAN_REG', y='CV',
                 color='QUALITY_FLAG', size='N_MONTHS',
                 hover_data=['SERIES_ID'],
                 title='Training Series Quality: Mean Registrations vs Coefficient of Variation',
                 labels={'MEAN_REG': 'Mean Monthly Registrations', 'CV': 'Coefficient of Variation'})
fig.add_hline(y=0.5, line_dash='dash', line_color='red',
              annotation_text='High Volatility Threshold')
fig.update_layout(template='plotly_white')
fig.show()

## 4. False Positive Identification Strategy

Combine multiple signals to classify each anomaly as **True Positive** or **Likely False Positive**:
1. **Similarity score** — high similarity to baseline = likely FP
2. **Training data quality** — flagged series = likely FP
3. **Deviation magnitude** — small deviations more likely FP
4. **Historical pattern** — one-off vs recurring

In [ ]:
df_classified = session.sql("""
    WITH anomalies AS (
        SELECT SERIES, TS, Y as ACTUAL, FORECAST, PERCENTILE,
               ABS(Y - FORECAST) as ABS_DEVIATION,
               CASE WHEN FORECAST > 0 THEN ABS(Y - FORECAST) / FORECAST ELSE 0 END as PCT_DEVIATION
        FROM TMR_ANOMALY_RESULTS
        WHERE IS_ANOMALY = TRUE
    ),
    series_quality AS (
        SELECT SERIES_ID, AVG(REGISTRATIONS) as MEAN_REG, STDDEV(REGISTRATIONS) as STD_REG,
               COUNT(*) as N_MONTHS,
               CASE WHEN AVG(REGISTRATIONS) > 0 THEN STDDEV(REGISTRATIONS) / AVG(REGISTRATIONS) ELSE 0 END as CV
        FROM TMR_SUBURB_MONTHLY
        WHERE TS < '2026-02-01'
        GROUP BY SERIES_ID
    ),
    suburb_profiles AS (
        SELECT SERIES_ID as SUBURB,
               'Suburb ' || SERIES_ID || ' avg ' || ROUND(AVG(REGISTRATIONS), 1) ||
               ' std ' || ROUND(STDDEV(REGISTRATIONS), 1) || ' months ' || COUNT(*) as PROFILE_TEXT
        FROM TMR_SUBURB_MONTHLY GROUP BY SERIES_ID
    ),
    baseline AS (
        SELECT 'Typical suburb avg ' || ROUND(AVG(REGISTRATIONS), 1) ||
               ' std ' || ROUND(STDDEV(REGISTRATIONS), 1) || ' stable pattern' as BL
        FROM TMR_SUBURB_MONTHLY
    )
    SELECT
        a.SERIES, a.TS, a.ACTUAL, ROUND(a.FORECAST, 0) as FORECAST,
        ROUND(a.PCT_DEVIATION, 2) as PCT_DEVIATION,
        ROUND(sq.CV, 3) as TRAINING_CV,
        sq.N_MONTHS as TRAINING_MONTHS,
        ROUND(AI_SIMILARITY(sp.PROFILE_TEXT, b.BL), 4) as SIMILARITY_SCORE,
        CASE
            WHEN AI_SIMILARITY(sp.PROFILE_TEXT, b.BL) > 0.92 AND a.PCT_DEVIATION < 0.5
                THEN 'LIKELY_FALSE_POSITIVE'
            WHEN sq.CV > 0.5 AND a.PCT_DEVIATION < 0.3
                THEN 'LIKELY_FALSE_POSITIVE'
            WHEN a.PCT_DEVIATION > 1.0
                THEN 'TRUE_ANOMALY'
            WHEN AI_SIMILARITY(sp.PROFILE_TEXT, b.BL) < 0.85
                THEN 'TRUE_ANOMALY'
            ELSE 'NEEDS_REVIEW'
        END as FP_CLASSIFICATION
    FROM anomalies a
    JOIN series_quality sq ON a.SERIES = sq.SERIES_ID
    JOIN suburb_profiles sp ON a.SERIES = sp.SUBURB
    CROSS JOIN baseline b
    ORDER BY FP_CLASSIFICATION, PCT_DEVIATION DESC
""").to_pandas()

fp_counts = df_classified['FP_CLASSIFICATION'].value_counts()
print('Anomaly Classification Results:')
for cls, cnt in fp_counts.items():
    print(f'  {cls}: {cnt}')
print(f'\nDetailed classification:')
df_classified[['SERIES','TS','ACTUAL','FORECAST','PCT_DEVIATION','TRAINING_CV','SIMILARITY_SCORE','FP_CLASSIFICATION']]

In [ ]:
fig = px.scatter(df_classified, x='SIMILARITY_SCORE', y='PCT_DEVIATION',
                 color='FP_CLASSIFICATION', size='ACTUAL',
                 hover_data=['SERIES', 'TRAINING_CV'],
                 title='Anomaly Classification: Similarity Score vs Deviation Magnitude',
                 labels={'SIMILARITY_SCORE': 'Similarity to Baseline (higher = more normal)',
                         'PCT_DEVIATION': 'Percentage Deviation from Forecast'},
                 color_discrete_map={'TRUE_ANOMALY': '#E22339',
                                    'LIKELY_FALSE_POSITIVE': '#339D37',
                                    'NEEDS_REVIEW': '#FFCC2C'})
fig.update_layout(template='plotly_white')
fig.show()

## 5. Retrain with Cleaned Training Data

Remove problematic series from training data and retrain the model.  
Compare results: fewer false positives, higher confidence in true anomalies.

In [ ]:
n_excluded = session.sql("""
    CREATE OR REPLACE VIEW TMR_TRAINING_DATA_CLEAN AS
    WITH problem_series AS (
        SELECT SERIES_ID
        FROM TMR_SUBURB_MONTHLY
        WHERE TS < '2026-02-01'
        GROUP BY SERIES_ID
        HAVING
            CASE WHEN AVG(REGISTRATIONS) > 0 THEN STDDEV(REGISTRATIONS) / AVG(REGISTRATIONS) ELSE 0 END > 0.5
            OR COUNT(DISTINCT TS) < 10
    )
    SELECT TS, SERIES_ID, REGISTRATIONS
    FROM TMR_SUBURB_MONTHLY
    WHERE TS < '2026-02-01'
      AND SERIES_ID NOT IN (SELECT SERIES_ID FROM problem_series)
      AND SERIES_ID IN (
          SELECT SERIES_ID FROM TMR_SUBURB_MONTHLY
          WHERE TS < '2026-02-01'
          GROUP BY SERIES_ID HAVING COUNT(DISTINCT TS) >= 8
      )
""").collect()

orig_count = session.sql('SELECT COUNT(DISTINCT SERIES_ID) FROM TMR_TRAINING_DATA').to_pandas().iloc[0,0]
clean_count = session.sql('SELECT COUNT(DISTINCT SERIES_ID) FROM TMR_TRAINING_DATA_CLEAN').to_pandas().iloc[0,0]
print(f'Original training series: {orig_count}')
print(f'Clean training series: {clean_count}')
print(f'Excluded: {orig_count - clean_count} problematic series')

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION tmr_anomaly_model_clean(
    INPUT_DATA => TABLE(TMR_TRAINING_DATA_CLEAN),
    SERIES_COLNAME => 'SERIES_ID',
    TIMESTAMP_COLNAME => 'TS',
    TARGET_COLNAME => 'REGISTRATIONS',
    LABEL_COLNAME => '',
    CONFIG_OBJECT => {'on_error': 'skip'}
)

In [ ]:
CREATE OR REPLACE TABLE TMR_ANOMALY_RESULTS_CLEAN AS
SELECT *
FROM TABLE(
    tmr_anomaly_model_clean!DETECT_ANOMALIES(
        INPUT_DATA => TABLE(TMR_TEST_DATA),
        SERIES_COLNAME => 'SERIES_ID',
        TIMESTAMP_COLNAME => 'TS',
        TARGET_COLNAME => 'REGISTRATIONS',
        CONFIG_OBJECT => {'prediction_interval': 0.95}
    )
)

In [ ]:
df_orig = session.sql("SELECT COUNT(*) as N FROM TMR_ANOMALY_RESULTS WHERE IS_ANOMALY = TRUE").to_pandas()
df_clean = session.sql("SELECT COUNT(*) as N FROM TMR_ANOMALY_RESULTS_CLEAN WHERE IS_ANOMALY = TRUE").to_pandas()

orig_anom = df_orig.iloc[0,0]
clean_anom = df_clean.iloc[0,0]

print('=== Model Comparison ===')
print(f'Original model anomalies:  {orig_anom}')
print(f'Clean model anomalies:     {clean_anom}')
print(f'Reduction:                 {orig_anom - clean_anom} ({100*(orig_anom - clean_anom)/orig_anom:.1f}%)')

df_compare = session.sql("""
    SELECT
        COALESCE(o.SERIES, c.SERIES) as SUBURB,
        o.IS_ANOMALY as ORIG_ANOMALY,
        c.IS_ANOMALY as CLEAN_ANOMALY,
        CASE
            WHEN o.IS_ANOMALY = TRUE AND c.IS_ANOMALY = TRUE THEN 'CONFIRMED'
            WHEN o.IS_ANOMALY = TRUE AND (c.IS_ANOMALY = FALSE OR c.IS_ANOMALY IS NULL) THEN 'REMOVED (likely FP)'
            WHEN (o.IS_ANOMALY = FALSE OR o.IS_ANOMALY IS NULL) AND c.IS_ANOMALY = TRUE THEN 'NEW DETECTION'
            ELSE 'BOTH NORMAL'
        END as STATUS,
        ROUND(o.Y, 0) as ACTUAL,
        ROUND(o.FORECAST, 0) as ORIG_FORECAST,
        ROUND(c.FORECAST, 0) as CLEAN_FORECAST
    FROM TMR_ANOMALY_RESULTS o
    FULL OUTER JOIN TMR_ANOMALY_RESULTS_CLEAN c ON o.SERIES = c.SERIES AND o.TS = c.TS
    WHERE o.IS_ANOMALY = TRUE OR c.IS_ANOMALY = TRUE
    ORDER BY STATUS, SUBURB
""").to_pandas()

status_counts = df_compare['STATUS'].value_counts()
print(f'\nComparison Breakdown:')
for s, n in status_counts.items():
    print(f'  {s}: {n}')
print(f'\n')
df_compare

In [ ]:
comparison_data = pd.DataFrame({
    'Model': ['Original', 'Clean (filtered)'],
    'Anomalies': [len(df_compare[df_compare['ORIG_ANOMALY']==True]),
                  len(df_compare[df_compare['CLEAN_ANOMALY']==True])]
})

fig = go.Figure()
fig.add_trace(go.Bar(name='Original Model', x=['Anomalies Detected'],
                     y=[comparison_data.iloc[0]['Anomalies']], marker_color='#E22339'))
fig.add_trace(go.Bar(name='Clean Model', x=['Anomalies Detected'],
                     y=[comparison_data.iloc[1]['Anomalies']], marker_color='#339D37'))
fig.update_layout(title='Model Comparison: Anomaly Count Before/After Training Data Cleanup',
                  barmode='group', template='plotly_white')
fig.show()

## 6. LLM-Powered Model Evaluation

Use Cortex AI to evaluate the remaining anomalies and provide confidence ratings

In [ ]:
confirmed = df_compare[df_compare['STATUS'] == 'CONFIRMED']
if len(confirmed) > 0:
    anomaly_text = confirmed[['SUBURB','ACTUAL','ORIG_FORECAST','CLEAN_FORECAST']].to_string(index=False)
else:
    anomaly_text = df_compare.head(10).to_string(index=False)

llm_result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-6',
        $$You are a data quality analyst reviewing anomaly detection results for QLD Transport registrations.

These anomalies were CONFIRMED by both the original and cleaned model (high confidence true positives):

{anomaly_text}

For each anomaly:
1. Rate confidence (HIGH/MEDIUM/LOW) that this is a genuine anomaly
2. Suggest a likely cause
3. Recommend action (INVESTIGATE / MONITOR / DISMISS)

Also provide:
4. Overall assessment of the false positive elimination process
5. Recommendations for improving the training pipeline$$
    ) as EVALUATION
""").to_pandas()

print(llm_result.iloc[0]['EVALUATION'])

## 7. Embedding-Based Suburb Clustering

Use `AI_EMBED` to create semantic embeddings of suburb registration profiles,  
then find clusters of similar suburbs. Anomalies in low-density clusters are more likely true positives.

In [ ]:
df_embed_sim = session.sql("""
    WITH suburb_profiles AS (
        SELECT
            SERIES_ID as SUBURB,
            'Suburb ' || SERIES_ID || ': monthly vehicle registrations averaging ' ||
            ROUND(AVG(REGISTRATIONS), 0) || ' per month, ranging from ' ||
            MIN(REGISTRATIONS) || ' to ' || MAX(REGISTRATIONS) ||
            ', with ' || COUNT(*) || ' months of data, standard deviation ' ||
            ROUND(STDDEV(REGISTRATIONS), 1) as PROFILE_TEXT
        FROM TMR_SUBURB_MONTHLY
        GROUP BY SERIES_ID
    ),
    anomalous AS (
        SELECT DISTINCT SERIES FROM TMR_ANOMALY_RESULTS_CLEAN WHERE IS_ANOMALY = TRUE
    ),
    all_pairs AS (
        SELECT
            a.SUBURB as ANOMALY_SUBURB,
            b.SUBURB as COMPARISON_SUBURB,
            ROUND(AI_SIMILARITY(a.PROFILE_TEXT, b.PROFILE_TEXT), 4) as PAIR_SIMILARITY
        FROM suburb_profiles a
        JOIN anomalous an ON a.SUBURB = an.SERIES
        JOIN suburb_profiles b ON a.SUBURB != b.SUBURB
        WHERE b.SUBURB NOT IN (SELECT SERIES FROM anomalous)
    )
    SELECT
        ANOMALY_SUBURB,
        COUNT(*) as N_SIMILAR,
        ROUND(AVG(PAIR_SIMILARITY), 4) as AVG_SIMILARITY_TO_NORMAL,
        ROUND(MAX(PAIR_SIMILARITY), 4) as MAX_SIMILARITY_TO_NORMAL,
        CASE
            WHEN AVG(PAIR_SIMILARITY) > 0.90 THEN 'MANY_SIMILAR_NORMALS (check FP)'
            WHEN AVG(PAIR_SIMILARITY) < 0.80 THEN 'UNIQUE_PATTERN (likely real)'
            ELSE 'MODERATE'
        END as CLUSTER_ASSESSMENT
    FROM all_pairs
    GROUP BY ANOMALY_SUBURB
    ORDER BY AVG_SIMILARITY_TO_NORMAL DESC
""").to_pandas()

print('Embedding Cluster Analysis of Remaining Anomalies:')
df_embed_sim

## Key Takeaways

| Technique | Purpose | Snowflake Feature |
|-----------|---------|------------------|
| Similarity Scores | Compare anomalous suburbs to baseline | `AI_SIMILARITY` |
| Training Data Quality | IQR/CV analysis, flag problematic series | SQL analytics |
| False Positive Classification | Multi-signal anomaly classification | `AI_SIMILARITY` + SQL |
| Clean Retraining | Retrain without noisy data | `SNOWFLAKE.ML.ANOMALY_DETECTION` |
| Model Comparison | Before/after anomaly counts | SQL JOIN |
| LLM Evaluation | AI-powered confidence rating | `CORTEX.COMPLETE` |
| Embedding Clustering | Find unique vs common patterns | `AI_EMBED` / `AI_SIMILARITY` |

**Result: Cleaner training data → fewer false positives → higher trust in alerts**